In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PORTFOLIO_VALUE = 100_000_000

project_root = Path.cwd().parent
portfolio_path = project_root / "config" / "portfolio.csv"

portfolio = pd.read_csv(portfolio_path)
portfolio

,ticker,instrument_name,asset_class,target_weight,currency,primary_risk_factor
0,SPY,SPDR S&P 500 ETF,Equity,0.20,USD,US Equity
1,QQQ,Invesco QQQ Trust,Equity,0.10,USD,US Technology
2,IWM,iShares Russell 2000 ETF,Equity,0.05,USD,US Small Cap
3,XLF,Financial Select Sector SPDR,Equity,0.05,USD,US Financials
4,TLT,iShares 20+ Year Treasury Bond ETF,Rates,0.12,USD,Long Treasury
5,IEF,iShares 7-10 Year Treasury Bond ETF,Rates,0.08,USD,Intermediate Treasury
6,SHY,iShares 1-3 Year Treasury Bond ETF,Rates,0.05,USD,Short Treasury
7,LQD,iShares Investment Grade Corporate Bond ETF,Credit,0.10,USD,Investment Grade Credit
8,HYG,iShares High Yield Corporate Bond ETF,Credit,0.08,USD,High Yield Credit
9,GLD,SPDR Gold Shares,Commodity,0.07,USD,Gold


In [8]:
required_columns = {'ticker', 'instrument_name', 'asset_class','target_weight','currency','primary_risk_factor'}

#check if any columns are missing

missing_columns = required_columns - set(portfolio.columns)  # converting it to a set for subraction purposes
if missing_columns:
     raise ValueError(f"Missing columns: {missing_columns}")

#check if there are any duplicate tickers
if portfolio["ticker"].duplicated().any():
    raise ValueError("Portfolio contains duplicate tickers.")

#check for missing values
if portfolio.isna().any().any():
    raise ValueError("Portfolio contains missing values.")

#check if there are negative values in target weight
if (portfolio['target_weight'] <= 0).any():
    raise ValueError("target weight cannot be negative or zero.")

#check if the target weights add upto one 
WEIGHT_TOLERANCE = 1e-8
total_weight = portfolio["target_weight"].sum()

if not np.isclose(
    total_weight,
    1.0,
    atol=WEIGHT_TOLERANCE,
    rtol=0.0,
):
    raise ValueError(
        f"Target weights must sum to 1. "
        f"Current total: {total_weight:.10f}"
    )






getting data using yf 

In [10]:
import yfinance as yf

START_DATE = "2015-01-01"
END_DATE = "2026-08-12"

tickers = portfolio['ticker'].tolist()

raw_data = yf.download(
    tickers=tickers,
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=True,
    actions=False,
    progress=False,
)

raw_data.head()

Price            Close                                               \
Ticker             GLD        HYG        IEF         IWM        LQD   
Date                                                                  
2015-01-02  114.080002  48.336548  82.037643  102.543106  80.124306   
2015-01-05  115.800003  47.888809  82.538132  101.172188  80.451866   
2015-01-06  117.120003  47.705399  83.092613   99.421898  80.779564   
2015-01-07  116.430000  48.002060  83.077232  100.646271  80.886528   
2015-01-08  115.940002  48.363537  82.738373  102.353409  80.625748   

Price                                                                ...  \
Ticker            QQQ        SHY         SPY        TLT         USO  ...   
Date                                                                 ...   
2015-01-02  94.561165  68.491570  169.687851  92.287514  159.119995  ...   
2015-01-05  93.174042  68.491570  166.623322  93.737198  150.320007  ...   
2015-01-06  91.924751  68.524002  165.053925  95.426071  144.399994  ...   
2015-01-07  93.109756  68.556381  167.110657  95.237663  146.960007  ...   
2015-01-08  94.891853  68.548325  170.076111  93.976395  148.399994  ...   

Price          Volume                                                       \
Ticker            IEF       IWM        LQD       QQQ        SHY        SPY   
Date                                                                         
2015-01-02  2028600.0  46133100  2523600.0  31314600  1735400.0  121465900   
2015-01-05  1521700.0  51141900  3218800.0  36521300  1214300.0  169632600   
2015-01-06  1890200.0  67446000  5313400.0  66205500   959400.0  209151400   
2015-01-07  1558000.0  32252100  1636600.0  37577400   768800.0  125346700   
2015-01-08  1671000.0  28361700  2156900.0  40212600   862500.0  147217800   

Price                                             
Ticker           TLT      USO      UUP       XLF  
Date                                              
2015-01-02   9432000  2638313  1887600  40511471  
2015-01-05   9789500  3937438  2855500  50770502  
2015-01-06  18331300  5279375  2262700  57454463  
2015-01-07   9762900  3916575  2384600  36287049  
2015-01-08   8055300  3338188  1685500  37995923  

[5 rows x 60 columns]

Cleaning the data before finding returns

In [ ]:
prices = raw_data['Close'].copy()
prices  = prices.reindex(columns=tickers)
for ticker in prices.columns:                      # trying to figure out how many prices are missing before we calculate returns we use a for and if  
    missing_count = prices[ticker].isna().sum()    # sstatment to return dates of when the data is missing

    if missing_count > 0:
        missing_dates = prices.index[
            prices[ticker].isna()
        ]

        print(f"{ticker}: {missing_count} missing price(s)")
        print(missing_dates)
        print()            


IEF: 1 missing price(s)
DatetimeIndex(['2026-07-22'], dtype='datetime64[ns]', name='Date', freq=None)

SHY: 2 missing price(s)
DatetimeIndex(['2026-07-21', '2026-07-31'], dtype='datetime64[ns]', name='Date', freq=None)

LQD: 3 missing price(s)
DatetimeIndex(['2026-07-21', '2026-07-22', '2026-07-31'], dtype='datetime64[ns]', name='Date', freq=None)



since we have only 6 missing days of data for over 11 years of market data we can just drop these rows.

calculating returns

In [38]:
returns = prices.pct_change(fill_method = None)
#print(returns.isna().sum())  #used to see how many na rows are there in each ticker
returns = returns.dropna()
print(returns)

Ticker           SPY       QQQ       IWM       XLF       TLT       IEF  \
Date                                                                     
2015-01-05 -0.018060 -0.014669 -0.013369 -0.021028  0.015708  0.006101   
2015-01-06 -0.009419 -0.013408 -0.017300 -0.015283  0.018017  0.006718   
2015-01-07  0.012461  0.012891  0.012315  0.010487 -0.001974 -0.000185   
2015-01-08  0.017745  0.019140  0.016962  0.014944 -0.013243 -0.004079   
2015-01-09 -0.008014 -0.006583 -0.009603 -0.013497  0.010952  0.004933   
...              ...       ...       ...       ...       ...       ...   
2026-08-05 -0.001997 -0.009049 -0.006430  0.002073  0.002173  0.000643   
2026-08-06 -0.001598 -0.003694 -0.005071 -0.003276 -0.005783 -0.003858   
2026-08-07  0.006115  0.011726  0.011098 -0.003633  0.002908  0.002367   
2026-08-10 -0.000297 -0.002987 -0.005239  0.003646 -0.008458 -0.004401   
2026-08-11 -0.003195 -0.003357  0.003367 -0.000173  0.001584  0.001186   

Ticker           SHY       LQD       

In [42]:
portfolio["target_market_value"] = (
    portfolio["target_weight"] * PORTFOLIO_VALUE
)

position_values = portfolio.set_index("ticker")[
    "target_market_value"
]

missing_exposures = set(returns.columns) - set(position_values.index)
missing_returns = set(position_values.index) - set(returns.columns)

if missing_exposures:
    raise ValueError(
        f"Returns exist without portfolio exposures: {missing_exposures}"
    )

if missing_returns:
    raise ValueError(
        f"Portfolio positions exist without returns: {missing_returns}"
    )

position_values = position_values.reindex(returns.columns)

position_pnl = returns * position_values
position_pnl.head()

Ticker,SPY,QQQ,IWM,XLF,TLT,IEF,SHY,LQD,HYG,GLD,USO,UUP
Date,,,,,,,,,,,,
2015-01-05,-361196.090557,-146690.569011,-66845.950271,-105138.308776,188500.144962,48805.719941,0.000000,40881.530497,-74103.537362,105540.045158,-276520.489662,10330.329979
2015-01-06,-188376.627979,-134081.386067,-86500.546986,-76415.477519,216205.305379,53743.063463,2367.631256,40732.150737,-30639.416194,79792.725564,-196913.688774,10309.030841
2015-01-07,249219.421254,128910.350201,61574.607294,52434.883464,-23692.631875,-1480.840116,2362.613786,13241.481652,49748.898181,-41239.899049,88643.106842,18518.450661
2015-01-08,354909.035659,191397.435710,84808.808551,74720.432596,-158920.566393,-32630.798930,-587.592320,-32240.268049,60243.564326,-29459.632718,48992.484922,20500.182166
2015-01-09,-160276.573670,-65829.880448,-48014.761250,-67486.092212,131428.709674,39467.078985,4135.333074,26545.788528,40153.299231,79696.374583,-72775.892593,-22457.701741


In [ ]:
portfolio_pnl = position_pnl.sum(axis=1)
portfolio_returns = portfolio_pnl/100000000
portfolio_returns .head()

Date
2015-01-05   -0.006364
2015-01-06   -0.003098
2015-01-07    0.005982
2015-01-08    0.005817
2015-01-09   -0.001154
dtype: float64

historical var and es calculation

In [59]:
historical_var = np.percentile(portfolio_returns,1)
#tail_returns  = []
#for daily_returns in portfolio_returns:
#   if daily_returns <= historical_var:
#        tail_returns.append(daily_returns)
#historical_es = np.mean(tail_returns)
tail = portfolio_returns <= historical_var
tail_returns = portfolio_returns[tail]
historical_es = tail_returns.mean()
print(f"Historcial var : {historical_var}")
print(f"Historcial es : {historical_es}")
len(portfolio_returns)
len(tail_returns)
tail_returns.max()






Historcial var : -0.015159461063341864
Historcial es : -0.024320205594918636


np.float64(-0.015189686959123433)

next we calculate rollling historical var 

In [ ]:
window_size = 252
for forecast in range(window_size, len(portfolio_returns)):
    